In [ ]:
import ee
import xarray as xr
import rioxarray
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
ee.Authenticate()
ee.Initialize(project="cs578-wildfire-prediction")

In [ ]:
# Define bounding box and dates
bbox = ee.Geometry.Polygon([
    [[-119.08580653031808, 34.710191016920575],
     [-119.08581587345162, 34.01019101696305],
     [-117.58651951593266, 34.01019101696305],
     [-117.5865288590662, 34.710191016920575]]
])

start_date = "2025-01-01"
end_date = "2025-02-01"

In [ ]:
#     # --- 4. Export as GeoTIFF ---
#     # Get the first image to check bands (for debugging)
# for i in range(time_delta.days + 1):
#     day = start_date + datetime.timedelta(days=i)
#     next_day = day + datetime.timedelta(days=1)
    
#     day = str(day)
#     next_day = str(next_day)
    
#     viirs_fire = ee.ImageCollection('NASA/LANCE/SNPP_VIIRS/C2') \
#         .filterDate(day, next_day) \
#         .filterBounds(bbox) \
#         .filter(ee.Filter.neq('confidence', 0)) \
#         .select('frp') \
#         .map(lambda img: img.toFloat()) \
#         .first()


#     # GRIDMET Weather: All bands
#     weather = ee.ImageCollection("IDAHO_EPSCOR/GRIDMET") \
#         .filterDate(day, next_day) \
#         .filterBounds(bbox) \
#         .map(lambda img: img.toFloat()) \
#         .first()

#     # https://developers.google.com/earth-engine/datasets/catalog/JAXA_GCOM-C_L3_LAND_LAI_V3
#     vegetation = ee.ImageCollection("JAXA/GCOM-C/L3/LAND/LAI/V3") \
#         .filterDate(day, next_day) \
#         .filterBounds(bbox) \
#         .select('LAI_AVE') \
#         .filter(ee.Filter.eq('SATELLITE_DIRECTION', 'D')) \
#         .map(lambda img: img.toFloat()) \
#         .first()
    
#     combined = viirs_fire.addBands(weather).addBands(vegetation)

#     # Export the entire collection as a stacked GeoTIFF
#     export_params = {
#         'image': combined,
#         'description': f"Fire prediction data for {str(day)}",
#         'folder': 'Fire_Pred',
#         'fileNamePrefix': str(day).replace("-", ""),
#         'region': bbox,
#         'scale': 375,  # 375m resolution
#         'crs': 'EPSG:4326',
#         'fileFormat': 'GeoTIFF',
#         'maxPixels': 1e9
#     }

#     # Start the export (check Tasks tab in GEE Console)
#     task = ee.batch.Export.image.toDrive(**export_params)
#     task.start()
#     print(day)

In [ ]:
import datetime
import glob
import os

def merge_to_netcdf_with_separate_vars(tif_folder, output_nc):
    """Merge daily GeoTIFFs into NetCDF with separate variables"""
    # Get all GeoTIFF files
    tif_files = sorted(glob.glob(os.path.join(tif_folder, '*.tif')))
    
    # Initialize lists to store data arrays
    variables = {}
    
    # Process each file
    for f in tif_files:
        try:
            # Extract date from filename
            date_str = os.path.basename(f).split('.')[0]
            date = pd.to_datetime(date_str)
            
            # Open the GeoTIFF
            ds = rioxarray.open_rasterio(f)
            
            # Get band names (assuming they're preserved in the GeoTIFF)
            if not variables:  # First file - initialize variables
                num_bands = len(ds.band)
                # Create default band names if not available
                band_names = ds.attrs.get('long_name', [f'band_{i}' for i in range(num_bands)])
                if isinstance(band_names, str):  # Handle case where it's a string
                    band_names = eval(band_names)  # Convert string to list
                
                for i, band_name in enumerate(band_names):
                    variables[band_name] = {
                        'data': [],
                        'attrs': {
                            'long_name': str(band_name),  # Set long_name to just the band name
                            'units': ds.attrs.get('units', 'unknown')
                        }
                    }
            
            # Append data for each band
            for i, band_name in enumerate(variables.keys()):
                band_data = ds.isel(band=i)
                variables[band_name]['data'].append(band_data.expand_dims(time=[date]))
                
        except Exception as e:
            print(f"Error processing {f}: {str(e)}")
            continue
    
    if not variables:
        raise ValueError("No valid data found in any GeoTIFF")
    
    # Create final dataset
    ds_out = xr.Dataset()
    for var_name, var_dict in variables.items():
        var_data = xr.concat(var_dict['data'], dim='time')
        var_data = var_data.rename({'x': 'lon', 'y': 'lat'})
        var_data = var_data.drop_vars(['band', 'spatial_ref'], errors='ignore')
        
        # Create DataArray with proper attributes
        da = xr.DataArray(
            data=var_data,
            dims=('time', 'lat', 'lon'),
            attrs=var_dict['attrs']  # Only includes the variable's own name
        )
        ds_out[var_name] = da
    
    # Global attributes
    ds_out.attrs = {
        'description': 'Combined dataset for fire prediction.',
        'created': datetime.date.today().isoformat()
    }
    
    # Save with compression
    encoding = {var: {'zlib': True, 'complevel': 5} for var in ds_out.data_vars}
    ds_out.to_netcdf(output_nc, encoding=encoding)
    print(f"Saved clean NetCDF to {output_nc}")

In [ ]:
merge_to_netcdf_with_separate_vars('../geotiffs', '../final_output_separate_vars.nc')

In [ ]:
with xr.open_dataset("../fire_pred_dataset.nc") as f:
    print(f['LAI_AVE'].mean())